# **Summarization Middleware**

The summarization middleware monitors message token counts and automatically summarizes older messages when thresholds are reached.

**Configuration options:**
- model: The language model to use for generating summaries.
- summary_prompt: Prompt template for generating summaries. There exist a default prompt template.
- trigger: One or more thresholds that trigger summarization.
    - `("messages", 50)`: Trigger summarization when 50 messages is reached
    - `("tokens", 3000)`: Trigger summarization when 3000 tokens is reached
    - `[("fraction", 0.8), ("messages", 100)]`: Trigger summarization either when 80% of model's max input tokens is reached or when 100 messages is reached (whichever comes first)
- keep: Context retention policy applied after summarization.
    - `("messages", 20)`: Keep the most recent 20 messages
    - `("tokens", 3000)`: Keep the most recent 3000 tokens
    - `("fraction", 0.3)`: Keep the most recent 30% of the model's max input tokens
- trim_tokens_to_summarize: Maximum tokens to keep when preparing messages for the summarization call. Pass `None` to skip trimming entirely. Default to `4000` tokens.

**Syntax:**
```python
agent = create_agent(
    model=chat_model,
    middleware=SummarizationMiddleware(
        model=summarization_model,
        summary_prompt=summary_prompt,
        # Trigger summarization when 70% of context is used
        trigger=("fraction", 0.7),
        # Keep the most recent 30% of messages in full
        keep=("fraction", 0.3),
        # No additional trimming before summarization
        trim_tokens_to_summarize=None,
    )
)
```

In [1]:
from langchain_groq import ChatGroq

# Setup API Key
f = open('keys/.groq_api_key.txt')
GROQ_API_KEY = f.read()

chat_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="openai/gpt-oss-20b", 
    temperature=1
)

In [2]:
SUMMARIZATION_PROMPT = """
Summarize the main thrust of this conversation. What have the human and assistant
discussed so far? Focus on key facts and requests.
<messages>
Messages to summarize:
{messages}
</messages>
"""

summarization_model = ChatGroq(
    api_key=GROQ_API_KEY, 
    model="llama-3.1-8b-instant", 
    temperature=1
)

In [3]:
SYSTEM_PROMPT = '''
You are a senior technical support specialist for electronic gadgets. Your goal is to identify and resolve user hardware or software issues efficiently.

RULES:
1. BREVITY: Limit every response to a maximum of 3 sentences. Get straight to the point.
2. METHODOLOGY: Always follow a diagnostic loop: 
   - Ask clarifying questions to isolate the root cause.
   - Suggest one clear, actionable step at a time.
   - Wait for the user's feedback before proceeding to the next step.
3. TONE: Professional, calm, and empathetic. Do not use jargon unless necessary.
4. SAFETY: If a suggestion involves risk (e.g., hardware disassembly), add a standard safety warning.
5. CONTEXT: Use the provided conversation history to avoid repeating previous troubleshooting steps.

If the user reports a successful fix, close the ticket politely and offer further assistance if needed.
'''

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model=chat_model,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            summary_prompt=SUMMARIZATION_PROMPT,
            trigger=("tokens", 100),    # Number of tokens we want our conversation to grow to before summarization
            keep=("messages", 1),       # Number of messages to keep after summarization - it deletes all the previous messages
        )
    ]
)

In [6]:
response = agent.invoke({"messages": "My laptop is running very slow."}, {"configurable" : {"thread_id" : "1"}})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

My laptop is running very slow.
================================== Ai Message ==================================

Could you let me know if your laptop is running slow during normal usage, or only when certain applications or programs are open?


In [7]:
response = agent.invoke({"messages": "When I run local LLMs from HuggingFace."}, {"configurable" : {"thread_id" : "1"}})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

The human has requested assistance with their slow laptop. The key points discussed so far are:

1. The user's laptop is experiencing performance issues.
2. The assistant has asked for clarification on whether the laptop is slow during normal usage or only when specific applications are open.
================================ Human Message =================================

When I run local LLMs from HuggingFace.
================================== Ai Message ==================================

It sounds like the slowdown happens when you start your local LLMs.  
Which model are you loading and how much RAM is allocated to the process?  
(If you’re not sure, check the memory usage in Task Manager or `top` before launching.)


In [8]:
response = agent.invoke({"messages": "I am Launching Qwen Model and 8GB RAM is allocated."}, {"configurable" : {"thread_id" : "1"}})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

Here's a summary of the key facts and requests discussed so far:

**Key Facts:**

1. The user's laptop is experiencing performance issues.
2. The slowdown occurs when the user runs local Large Language Models (LLMs) from HuggingFace.

**Key Requests:**

1. The assistant asked for clarification on when the slowdown happens (during normal usage or when specific applications are open).
2. The assistant asked the user to provide more information on the specific LLM model being loaded and the amount of RAM allocated to the process.
================================ Human Message =================================

I am Launching Qwen Model and 8GB RAM is allocated.
================================== Ai Message ==================================

Could you let me know which device you’re running the Qwen model on (CPU or GPU) and whether you observe high CPU/GPU usa

### **Retrieve the Agent State**

In [11]:
# 1. Define the config with your thread_id
config = {"configurable": {"thread_id": "1"}}

# 2. Retrieve the snapshot
snapshot = agent.get_state(config)

# 3. Access the full message history
all_messages = snapshot.values.get("messages", [])

# 4. Print or inspect the messages
for msg in all_messages:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

Here's a summary of the key facts and requests discussed so far:

**Key Facts:**

1. The user's laptop is experiencing performance issues.
2. The slowdown occurs when the user runs local Large Language Models (LLMs) from HuggingFace.

**Key Requests:**

1. The assistant asked for clarification on when the slowdown happens (during normal usage or when specific applications are open).
2. The assistant asked the user to provide more information on the specific LLM model being loaded and the amount of RAM allocated to the process.
================================ Human Message =================================

I am Launching Qwen Model and 8GB RAM is allocated.
================================== Ai Message ==================================

Could you let me know which device you’re running the Qwen model on (CPU or GPU) and whether you observe high CPU/GPU usa

### **Managing Long Conversations**

With the help of checkpointer, we've got an agent that maintains a list of messages so that it can remember what happened in our conversation till date.

This works well for the shorter conversations. After a while the list of messages become longer, overflowing agents context window. This leads to slower application, and increasing cost. 

There are two ways in middleware we can solve this problem:
1. Summarizing the conversation
2. Trimming or deleting the messages


#### **Summarize Messages**
Covered Above

### **Trimming or Deleting Messages**

Run once before or after agent run:
- @before_agent
- @after_agent

Runs multiple times, before or after each model call
- @before_model
- @after_model

#### **Let's say we want to remove all the ToolMessage before the start of an agent run**

Assuming ToolMessages doesn't contain enough information and they are just clutering the context window. 

In [ ]:
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage, RemoveMessage

from langgraph.runtime import Runtime

@before_agent
def custom_trim_messages(runtime: Runtime) -> dict | None:
    "Remove all the Tool Message from the state"
    messages = runtime["messages"]

    tool_messages = [msg for msg in messages if isinstance(msg, ToolMessage)]

    return {"messages" : RemoveMessage(id=msg.id) for msg in tool_messages}